In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/CV Bootcamp'
OUT = DATA + '/outputs'
import os, sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
os.makedirs(OUT, exist_ok=True)
sys.path.append(DATA)
from model import ResNet18
torch.manual_seed(0); np.random.seed(0)
dev = 'cuda'

Mounted at /content/drive


In [ ]:
X = torch.tensor(np.load(f'{DATA}/data/train_images.npy')).permute(0, 3, 1, 2).float().div(255)
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
X = ((X - mean) / std).to(dev)
lab = pd.read_csv(f'{DATA}/data/labels.csv').set_index('index').sort_index()
tr = np.loadtxt(f'{DATA}/data/train_idx.txt', dtype=int)
va = np.loadtxt(f'{DATA}/data/val_idx.txt', dtype=int)
noisy = torch.tensor(lab.noisy_label.values).to(dev)
clean = torch.tensor(lab.clean_label.values).to(dev)
tr_t, va_t = torch.tensor(tr).to(dev), torch.tensor(va).to(dev)
pos = torch.zeros(50000, dtype=torch.long, device=dev)
pos[tr_t] = torch.arange(len(tr), device=dev)

In [ ]:
def augment(x):
    n = x.size(0)
    x = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (n,), device=dev), torch.randint(0, 9, (n,), device=dev)
    r = torch.arange(32, device=dev)
    x = x[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
          (i[:, None] + r)[:, None, :, None], (j[:, None] + r)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    return x

@torch.no_grad()
def probs(idx):
    net.eval()
    with torch.autocast('cuda', dtype=torch.float16):
        return torch.cat([F.softmax(net(X[idx[k:k + 1000]]).float(), 1) for k in range(0, len(idx), 1000)])

In [ ]:
E, B, LR = 60, 128, 0.1
net = ResNet18(10).to(dev)
u = torch.nn.Parameter(torch.randn(len(tr), 10, device=dev) * 1e-8)
v = torch.nn.Parameter(torch.randn(len(tr), 10, device=dev) * 1e-8)
opt = torch.optim.SGD([{'params': net.parameters()},
                       {'params': [u, v], 'lr': 0.2 * B, 'momentum': 0, 'nesterov': False, 'weight_decay': 0}],
                      lr=LR, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
P = np.zeros((E, len(tr), 10), dtype=np.float16)
V = np.zeros(E)
for ep in range(E):
    net.train()
    perm = tr_t[torch.randperm(len(tr_t), device=dev)]
    for s in range(0, len(perm), B):
        b = perm[s:s + B]
        with torch.autocast('cuda', dtype=torch.float16):
            logits = net(augment(X[b]))
        pr = F.softmax(logits.float(), 1)
        yb = F.one_hot(noisy[b], 10).float()
        us, vs = u[pos[b]] ** 2 * yb, v[pos[b]] ** 2 * (1 - yb)
        q = torch.clamp(pr + us - vs.detach(), min=1e-4)
        q = q / q.sum(1, keepdim=True)
        ce = -(yb * q.log()).sum(1).mean()
        mse = ((F.one_hot(pr.argmax(1), 10).float() + us.detach() - vs - yb) ** 2).sum(1).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(ce + mse).backward(); scaler.step(opt); scaler.update()
        with torch.no_grad():
            u.clamp_(-1, 1); v.clamp_(-1, 1)
    sched.step()
    P[ep] = probs(tr_t).cpu().numpy()
    pv = probs(va_t).argmax(1)
    V[ep] = (pv == noisy[va_t]).float().mean().item()
    print(ep + 1, f'val_noisy {V[ep]:.4f}', f'val_clean {(pv == clean[va_t]).float().mean().item():.4f}')
np.save(f'{OUT}/t3B_train_probs.npy', P)
np.save(f'{OUT}/t3B_val_noisy.npy', V)
torch.save({'u': u.detach().cpu(), 'v': v.detach().cpu()}, f'{OUT}/t3B_uv.pt')

1 val_noisy 0.1024 val_clean 0.0874
2 val_noisy 0.2026 val_clean 0.2322
3 val_noisy 0.2938 val_clean 0.3626
4 val_noisy 0.3302 val_clean 0.4200
5 val_noisy 0.3792 val_clean 0.4958
6 val_noisy 0.3754 val_clean 0.5130
7 val_noisy 0.4142 val_clean 0.5494
8 val_noisy 0.4172 val_clean 0.5690
9 val_noisy 0.4498 val_clean 0.6014
10 val_noisy 0.4454 val_clean 0.6110
11 val_noisy 0.4592 val_clean 0.6294
12 val_noisy 0.4740 val_clean 0.6512
13 val_noisy 0.4666 val_clean 0.6506
14 val_noisy 0.4864 val_clean 0.6598
15 val_noisy 0.4852 val_clean 0.6652
16 val_noisy 0.4674 val_clean 0.6482
17 val_noisy 0.4810 val_clean 0.6368
18 val_noisy 0.5104 val_clean 0.7124
19 val_noisy 0.4948 val_clean 0.7094
20 val_noisy 0.5050 val_clean 0.7196
21 val_noisy 0.5036 val_clean 0.7132
22 val_noisy 0.5014 val_clean 0.6948
23 val_noisy 0.4990 val_clean 0.6894
24 val_noisy 0.5096 val_clean 0.7116
25 val_noisy 0.5012 val_clean 0.7198
26 val_noisy 0.5214 val_clean 0.7518
27 val_noisy 0.5186 val_clean 0.7332
28 val_noi

In [ ]:
y = lab.noisy_label.values[tr]
wrong = y != lab.clean_label.values[tr]
k = wrong.sum()
def score(p):
    pg = p[np.arange(len(tr)), y]
    o = p.copy(); o[np.arange(len(tr)), y] = 0
    return np.log(o.max(1) + 1e-12) - np.log(pg + 1e-12)
def check(s, name):
    print(f'{name:30s} precision@k {wrong[np.argsort(-s)[:k]].mean():.4f}  AUROC {roc_auc_score(wrong, s):.4f}')
Pf = P.astype(np.float32)
peak = V.argmax() + 1
yh = F.one_hot(torch.tensor(y), 10).float().numpy()
uu, vv = u.detach().cpu().numpy() ** 2, v.detach().cpu().numpy() ** 2
S_all = score(Pf.mean(0))
check(S_all, 'avg epochs 1-60')
check(score(Pf[:peak].mean(0)), f'avg epochs 1-{peak} (noisy peak)')
check((uu * yh + vv * (1 - yh)).sum(1), 'SOP noise size')
pd.DataFrame({'index': tr, 'score': S_all}).to_csv(f'{OUT}/t3B_noise_scores.csv', index=False)

avg epochs 1-60                precision@k 0.8629  AUROC 0.9517
avg epochs 1-60 (noisy peak)   precision@k 0.8629  AUROC 0.9517
SOP noise size                 precision@k 0.7775  AUROC 0.9097
